In [2]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 1. 데이터 가져오기

In [39]:
import pandas as pd
df = pd.read_csv('C:/ai_x/source/01_python/data/전국평당분양가격(결측보완).csv', encoding='cp949')
df

,지역명,연도,월,평당분양가격
0,서울,2013,12,18189.0
1,부산,2013,12,8111.0
2,대구,2013,12,8080.0
3,인천,2013,12,10204.0
4,광주,2013,12,6098.0
...,...,...,...,...
2171,전북,2024,8,12058.2
2172,전남,2024,8,13120.8
2173,경북,2024,8,13827.0
2174,경남,2024,8,13252.8


# 2. 지역명의 라벨 인코딩
- 지역명을 라벨인코딩한 지역명2
- 분석할 경우 원핫인코딩까지 할 것을 추천

In [40]:
from tensorflow.keras.utils import to_categorical # 분류분석시 원핫인코딩
from tensorflow.keras.models import Sequential # 모델 생성
from tensorflow.keras.layers import Dense,Input
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from pandas import Series, DataFrame


17

In [42]:
# X, y 분리
X_df = df[['지역명','연도','월']].values
y_df = df[['평당분양가격']].values
print(X_df[:3])
print(y_df[:3])
# 지역명 라벨인코딩, 원핫인코딩
le = LabelEncoder()
X_df['지역명2'] = le.fit_transform(df['지역명'])
one_hot_encoded = to_categorical(X_df['지역명2'])
one_hot_encoded =  DataFrame(one_hot_encoded, columns=[f'R{i}' for i in range(len(one_hot_encoded.T))])
X_df = pd.concat([X_df, one_hot_encoded], axis=1)
X_df.head(5)

[['서울' 2013 12]
 ['부산' 2013 12]
 ['대구' 2013 12]]
[[18189.]
 [ 8111.]
 [ 8080.]]


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

# 3. normalization 스케일 조정
- 입력변수(지역명2, 연도, 월)와 타겟변수(평당분양가격) 따로 스케일 조정(MinMaxScaler 이용)
- 지역명2n, 연도n, 월n필드 추가

In [36]:
from sklearn.preprocessing import MinMaxScaler
scaler_x = MinMaxScaler() # x_data를 정규화시킬 객체
X_df['지역명2n'] = scaler_x.fit_transform(np.array(X_df['지역명2']).reshape(-1,1))
X_df['연도n'] = scaler_x.fit_transform(np.array(X_df['연도']).reshape(-1,1))
X_df['월n'] = scaler_x.fit_transform(np.array(X_df['월']).reshape(-1,1))

scaler_y = MinMaxScaler()
scaled_y_df = scaler_y.fit_transform(np.array(y_df).reshape(-1,1))

ValueError: could not convert string to float: '제주'

# 4. standardization 스케일 조정
- 입력변수와 타겟변수 따로 스케일 조정(StandardScaler 이용)
- 지역명2s, 연도s, 월s필드 추가